## Data descriptions

---
In this competition you are predicting the probability that an online transaction is fraudulent, as denoted by the binary target isFraud.

The data is broken into two files identity and transaction, which are joined by TransactionID. Not all transactions have corresponding identity information.

Categorical Features - Transaction
ProductCD
card1 - card6
addr1, addr2
P_emaildomain
R_emaildomain
M1 - M9
Categorical Features - Identity
DeviceType
DeviceInfo
id_12 - id_38
The TransactionDT feature is a timedelta from a given reference datetime (not an actual timestamp).

You can read more about the data from this post by the competition host.

Files
train_{transaction, identity}.csv - the training set
test_{transaction, identity}.csv - the test set (you must predict the isFraud value for these observations)
sample_submission.csv - a sample submission file in the correct format

---

I see many questions regarding data description, so it maybe a better idea to open a thread for discussion. The following is a bit more details about it:

Transaction Table *

TransactionDT: timedelta from a given reference datetime (not an actual timestamp)

TransactionAMT: transaction payment amount in USD

ProductCD: product code, the product for each transaction

card1 - card6: payment card information, such as card type, card category, issue bank, country, etc.

addr: address

dist: distance

P_ and (R__) emaildomain: purchaser and recipient email domain

C1-C14: counting, such as how many addresses are found to be associated with the payment card, etc. The actual meaning is masked.

D1-D15: timedelta, such as days between previous transaction, etc.

M1-M9: match, such as names on card and address, etc.

Vxxx: Vesta engineered rich features, including ranking, counting, and other entity relations.

Categorical Features: ProductCD card1 - card6 addr1, addr2 P_emaildomain R_emaildomain M1 - M9

Identity Table *

Variables in this table are identity information – network connection information (IP, ISP, Proxy, etc) and digital signature (UA/browser/os/version, etc) associated with transactions. They're collected by Vesta’s fraud protection system and digital security partners. (The field names are masked and pairwise dictionary will not be provided for privacy protection and contract agreement)

Categorical Features: DeviceType DeviceInfo id_12 - id_38

---

In [13]:
import pandas as pd
import networkx as nx

In [14]:
train_identity = pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')


In [15]:
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [16]:
train_transaction.head().iloc[:, :12]


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0


# Plan for making a graph
- Many nodes: eamils, transaction id, addresses, devices, IP information (identify all of these in the two data tables and work on relating the in networkx)
- Make all edges connections from transaction to other node types

``` 
Basic example to build from
import networkx as nx

G = nx.Graph()

for _, row in df.iterrows():
    t = f"T_{row['TransactionID']}"
    card = f"C_{row['card1']}"
    
    G.add_edge(t, card)
```

In [17]:
train_identity[['id_01', 'id_02']]

,id_01,id_02
0,0.0,70787.0
1,-5.0,98945.0
2,-5.0,191631.0
3,-5.0,221832.0
4,0.0,7460.0
...,...,...
144228,-15.0,145955.0
144229,-5.0,172059.0
144230,-20.0,632381.0
144231,-5.0,55528.0


In [32]:
def create_graph_edges_for_similar_cols(
        target_graph: nx.Graph,
        target_df: pd.DataFrame, 
        primary_node_col: str,
        primary_node_prefix: str,
        target_col_names: list[str], 
        ref_node_prefix: str, 
        ref_node_suffix: list[str] = None
    ) -> int:
    """
        This is a utility function for iterating similar columns that might need to be added to a graph in bulk. It will edit the graph passed in the argument directly.
        Retruns the size of the updated graph.

        graph: An instance of a networkx graph that needs to have connections added to it
        target_df: The data frame containing all reference columns for the mappings
        primary_node_col: The name of the column that will server as the primary node that should as the basis for many connections (ex. Transactions in a case where you want to connect many other attributes about that transaction to it),
        primary_node_prefix: A string that is the prefix for the primary node that will be used on the Graph (ex. if the primary node is Transactions you may want to use something like 'T' or 'Tr')
        target_col_names: a list of strings containing the names of columns in the target DataFrame that need to have relationships created for
        graph_label_prefix: the desired prefix for the target column to be associated with in the graph (ex if the columns are all related to credit card info you may want to use somehting like 'C')
        graph_label_suffix: this is the method for appending unique identifiers to the graph_label_prefix when there are multiple cols (once again for card info you may want something like C1, C2 ...)
                            If nothing is passed to this argument it will fill with 1 - n where n is the len of the  target_col_names. If you want custom functionality pass list of the suffix values in order.
    """
    tdf = target_df.copy() # ensure that the orginal df is not edited by this

    if ref_node_suffix != None:
        assert len(set(ref_node_suffix)) == len(ref_node_suffix), 'Graph label suffix values must be unique.'
    else: 
        ref_node_suffix = list(range(1, len(target_col_names) + 1)) # default functionality for using digits as the suffix

    for col_name, suffix in zip(target_col_names, ref_node_suffix):
        mask = tdf[col_name].notna()
        
        target_data = tdf[mask]

        connection_tuple_mappings = zip(
            f'{primary_node_prefix}_' + target_data[primary_node_col].astype('str'), 
            f'{ref_node_prefix}{suffix}_' + target_data[col_name].astype('str') # puts creates prefix of C1, C2, ... to C6 so each card identifier is unique
        )

        target_graph.add_edges_from(connection_tuple_mappings)
    
    return target_graph.size()


In [33]:
# get all connections created
# zipping the target cols as string and using add_edges_from() is more efficent so use this

# instantiate the graph
transactions_graph = nx.Graph()

# get all relevant fields that need to be added to the graph (all of this centers on transaction IDs and attributes that relate to those)
# all card data (card1 - card6)
card_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']

# create card connections
create_graph_edges_for_similar_cols(
    target_graph= transactions_graph,
    target_df=train_transaction,
    primary_node_col='TransactionID',
    primary_node_prefix='Tr',
    target_col_names=card_cols,
    ref_node_prefix='C'
)

addr_cols = ['addr1', 'addr2']
# address info (addr1 and addr2 )
create_graph_edges_for_similar_cols(
    target_graph= transactions_graph,
    target_df=train_transaction,
    primary_node_col='TransactionID',
    primary_node_prefix='Tr',
    target_col_names=addr_cols,
    ref_node_prefix='Addr'
)

# identity information is next
# Looks like id_02 is the only thing that can reasonably be used without introducing a bunch of nodes with artificially high degree measures
create_graph_edges_for_similar_cols(
    target_graph= transactions_graph,
    target_df=train_identity,
    primary_node_col='TransactionID',
    primary_node_prefix='Tr',
    target_col_names=['id_02'],
    ref_node_prefix='Id'
)



4715875

In [34]:
# inspecting basic graph attributes
print(f'Number of edges: {transactions_graph.number_of_edges()}')
print(f'Number of ndoes:  {transactions_graph.number_of_nodes()}')

Number of edges: 4715875
Number of ndoes:  720895


In [36]:
from collections import Counter

# looking at the nodes by each column from the original data set
Counter(n.split('_')[0] for n in transactions_graph.nodes)

Counter({'Tr': 590540,
         'Id1': 115655,
         'C1': 13553,
         'C2': 500,
         'Addr1': 332,
         'C5': 119,
         'C3': 114,
         'Addr2': 74,
         'C4': 4,
         'C6': 4})

In [ ]:
# analyzing the degree structure of the nodes

In [39]:
nodes_sorted_by_degree = sorted(list(transactions_graph.degree), key=lambda x: x[1], reverse=True)
nodes_sorted_by_degree[:20]

[('C3_150.0', 521287),
 ('Addr2_87.0', 520481),
 ('C6_debit', 439938),
 ('C4_visa', 384767),
 ('C5_226.0', 296546),
 ('C4_mastercard', 189217),
 ('C6_credit', 148986),
 ('C5_224.0', 81513),
 ('C5_166.0', 57140),
 ('C3_185.0', 56346),
 ('C2_321.0', 48935),
 ('Addr1_299.0', 46335),
 ('C2_111.0', 45191),
 ('Addr1_325.0', 42751),
 ('Addr1_204.0', 42020),
 ('C2_555.0', 41995),
 ('Addr1_264.0', 39870),
 ('C2_490.0', 38145),
 ('C5_102.0', 29105),
 ('Addr1_330.0', 26287)]

- It looks like C4 and C6 values may not be very useful here since they are generic categorical classifications that are likely to be shared and connect too many values.
- Will coninue to explore non-transaction nodes for high connectivity

In [40]:
nodes_to_explore = []

for pair in nodes_sorted_by_degree:
    non_interesting_nodes = ['Tr', 'C4', 'C6']
    node = pair[0]

    if node[:2] in non_interesting_nodes: continue

    nodes_to_explore.append(pair)


In [42]:
nodes_to_explore[:20]

[('C3_150.0', 521287),
 ('Addr2_87.0', 520481),
 ('C5_226.0', 296546),
 ('C5_224.0', 81513),
 ('C5_166.0', 57140),
 ('C3_185.0', 56346),
 ('C2_321.0', 48935),
 ('Addr1_299.0', 46335),
 ('C2_111.0', 45191),
 ('Addr1_325.0', 42751),
 ('Addr1_204.0', 42020),
 ('C2_555.0', 41995),
 ('Addr1_264.0', 39870),
 ('C2_490.0', 38145),
 ('C5_102.0', 29105),
 ('Addr1_330.0', 26287),
 ('C5_117.0', 25941),
 ('Addr1_315.0', 23078),
 ('C2_583.0', 21803),
 ('Addr1_441.0', 20827)]

In [9]:
max(nx.connected_components(transactions_graph), key=len)

{'Tr3027247',
 'Id1506675.0',
 'Tr3517133',
 'Tr3222411',
 'Tr3554750',
 'Tr3485637',
 'Id154755.0',
 'Tr3196319',
 'Tr3526392',
 'Tr3128844',
 'Tr3060474',
 'Tr3459293',
 'Tr3489270',
 'Id123480.0',
 'Tr3380852',
 'Tr3559754',
 'Tr3021120',
 'Tr3009041',
 'Tr3198508',
 'Tr3341075',
 'Tr3142742',
 'Tr3442970',
 'Tr2997350',
 'Tr3128850',
 'Tr2991797',
 'Id1343575.0',
 'Tr3053859',
 'Tr3467576',
 'Tr3483581',
 'Tr3470403',
 'Tr3305281',
 'Tr3306006',
 'Id1152682.0',
 'Tr3008633',
 'Id1166207.0',
 'Tr3110890',
 'Tr3384589',
 'Tr3552607',
 'Tr3100226',
 'Tr3062326',
 'Id1354301.0',
 'Tr3397424',
 'Tr3323546',
 'Tr3539008',
 'Tr3286691',
 'Tr3500605',
 'Tr3047150',
 'Id1152712.0',
 'Tr3388842',
 'Tr3228353',
 'Tr3083395',
 'Tr3210827',
 'Tr3339028',
 'Tr3467320',
 'Tr3060962',
 'Tr3088702',
 'Tr3409400',
 'Tr3566663',
 'Tr3427559',
 'Tr3222891',
 'Id1201745.0',
 'Id1181529.0',
 'Tr3146068',
 'Tr3294013',
 'Id156047.0',
 'Tr3362479',
 'Tr3508736',
 'Tr3055206',
 'Tr3116393',
 'Tr3489402',
 

In [10]:
print(f'Number of Edges: {transactions_graph.number_of_edges():,.0f}')
print(f'Number of Nodes: {transactions_graph.number_of_nodes():,.0f}')

Number of Edges: 1,781,080
Number of Nodes: 720,154


In [11]:
# taking a look at degeree centrality
top_degrees = dict(sorted(list(transactions_graph.degree), key=lambda x: x[1], reverse=True)[:10])

### Degree Centrality Notes

- All of the top ten most connected nodes are address nodes.

- This isn't too alarming, many businesses would have one address and could have a large volume of transaction traffic.
- The highest degree value is noticably higher than the rest of the values, which does seem suspicious, however this alone is not enough to definitively flag it.
- Will take a look at this valu addr2 == 87 to see if any other insights can be gained from it.

In [12]:
def get_fraud_ratio(target_pairs: dict):
    for key, val in target_pairs.items():
        target_col = 'addr' + key[4]
        lookup_val = float(key[5:])

        fraud_count = train_transaction[train_transaction[target_col] == lookup_val].isFraud.sum()
        total_count = train_transaction[train_transaction[target_col] == lookup_val].isFraud.count()

        print(f'Degree centrality for {target_col} == {lookup_val}: {val}')
        print(f'Fraud ratio for {target_col} == {lookup_val}: {(fraud_count / total_count) * 100:.4f}%')      
        print('---------------------------') 


get_fraud_ratio(top_degrees)


Degree centrality for addr2 == 87.0: 520481
Fraud ratio for addr2 == 87.0: 2.3972%
---------------------------
Degree centrality for addr1 == 299.0: 46335
Fraud ratio for addr1 == 299.0: 2.1258%
---------------------------
Degree centrality for addr1 == 325.0: 42751
Fraud ratio for addr1 == 325.0: 2.5426%
---------------------------
Degree centrality for addr1 == 204.0: 42020
Fraud ratio for addr1 == 204.0: 2.6654%
---------------------------
Degree centrality for addr1 == 264.0: 39870
Fraud ratio for addr1 == 264.0: 1.8259%
---------------------------
Degree centrality for addr1 == 330.0: 26287
Fraud ratio for addr1 == 330.0: 3.1955%
---------------------------
Degree centrality for addr1 == 315.0: 23078
Fraud ratio for addr1 == 315.0: 1.7809%
---------------------------
Degree centrality for addr1 == 441.0: 20827
Fraud ratio for addr1 == 441.0: 2.5592%
---------------------------
Degree centrality for addr1 == 272.0: 20141
Fraud ratio for addr1 == 272.0: 2.8598%
---------------------

In [13]:
# list(nx.connected_components(transactions_graph))


### Comments
- It appears that all of the high degree centrality nodes have similar fraud rates hovering in that 2-3%.

- Degree centrality is likely just not a good identifying measure here.
- I will continue to explore other options.

### Exploring connections between components

In [14]:
len_sorted_comps = sorted(nx.connected_components(transactions_graph), key=len, reverse=True)
[len(c) for c in len_sorted_comps]

[716347,
 35,
 26,
 25,
 22,
 21,
 19,
 19,
 19,
 17,
 17,
 16,
 15,
 15,
 15,
 15,
 14,
 14,
 14,
 13,
 13,
 13,
 13,
 13,
 13,
 13,
 13,
 13,
 13,
 12,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 10,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 8,
 8,
 8,
 8,
 8,
 8,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5

### Issues
- There is obviously an issues here where since one of the componenets is massive and there is a steep drop to the next one.
- Will look at ways to reduce this next and hopefully achieve more meaningful compoonents.

In [15]:
# examing the large component only to see why it is so massive
largest_comp = len_sorted_comps[0]

# getting the degrees for the largest component to see how connected they are
largest_degrees = sorted(list(transactions_graph.degree(largest_comp)), key=lambda x: x[1], reverse=True)


In [16]:
# trimming the graph down to only ndoes with degrees of 150 or lower
trimmed_graph = transactions_graph.copy()
for node, deg in transactions_graph.degree():
    if not node.startswith('Tr') and deg > 75: # if it isn't a transaction and has too high of degree, clip it
        trimmed_graph.remove_node(node)


In [17]:
# sorted([len(c) for c in list(nx.connected_components(trimmed_graph))], reverse=True)

con_comps_trimmed = sorted([c for c in list(nx.connected_components(trimmed_graph))], reverse=True, key=len)

In [18]:
len(con_comps_trimmed[0])

37516

### Looking at alternative clustering methods for better results

In [19]:
common_bridge_attributes = []

for value in list(nx.ego_graph(transactions_graph, 'Tr3474748', radius=2)):
    if not value.startswith('Tr'):
        print(value)
        common_bridge_attributes.append(value)

C17487
Addr1191.0
Addr287.0
Id141467.0


In [20]:
len(list(nx.ego_graph(transactions_graph, 'Tr3474748', radius=2)))

520485

In [21]:
520485 / 4 # ratio of total neighbors to the bridges

# maybe a ratio between 15 and 8 could be interesting

130121.25

- Since transactions can only be connected indirectly, and transacations dominate the negibors wihtin radius 2 of this transaction, thsee attributed must be very common for many transactions.
- It might be worth traversing to all transactions and seeing if there is a smaller number of neighbors connected by differnent bridges.

In [22]:
def find_uncommon_bridges(
        target_graph: nx.Graph, 
        start_point: str, 
        radius: int, 
        primary_prefix: str,
        interesting_finds: list[tuple] = [] # initialized as empty list and accumulated over recurive calls
    ) -> list[tuple]:

    # get the root neighbors list
    root_neighbors = list(nx.ego_graph(target_graph, start_point, radius=radius))
    
    key_transaction, size_of_cluster = start_point, len(root_neighbors)

    # find the common and uncommon bridges
    root_neighbor_bridges = set()
    transactions = set()
    uncommon_bridges = set()
    for node in root_neighbors:
        if not node.startswith(primary_prefix):
            root_neighbor_bridges.add(node)
            continue

        transactions.add(node)
        
    # get ratio of total cluster to the bridge nodes
    ratio = size_of_cluster / len(root_neighbor_bridges)

    # determine if bridges are uncommon by falling in this range
    if ratio <= 15 and ratio >= 8:
        print('found interesting result')
        uncommon_bridges = root_neighbor_bridges.copy()
        
        # append tuple of findings to interesting finds since there is a new observation
        interesting_finds.append((key_transaction, size_of_cluster, uncommon_bridges))
        return interesting_finds # this is the base case to end the BFS

    # if no uncommon bridge is found keep searching all neighbors recursively and updating the interesting finds
    for trans_node in transactions:
        found_bfs = find_uncommon_bridges(target_graph, trans_node, radius, primary_prefix, interesting_finds)
        if len(found_bfs): 
            print('Found interesting result')
            return interesting_finds + found_bfs
        
        # interesting_finds = interesting_finds +
        
    return interesting_finds
    # return a tupel of (key_transaction, size of cluster, uncommon_bridges)
    

In [23]:
intersting_clusters = find_uncommon_bridges(transactions_graph, 'Tr3474748', 2, 'Tr')

KeyboardInterrupt: 

### Creating mappings of components and transactions
- Since the primary interest is transactions components, this provides a map that links together any transaction that are connected through things like similar card use etc.
- Note that these connnections can be inderect where one transaction connects directly through a shared card, and then is further indirectly connected to a third transaction through an attribute shared by the second and third transactions only.

In [97]:
comp_mapppings = {}

for i, comp in enumerate(len_sorted_comps):
    for node in comp:
        comp_mapppings[node] = {
            'comp_id': i,
            'comp_size' : len(len_sorted_comps[i] )
        }

In [151]:
# [{attr: comp_mapppings[attr]} for attr in comp_mapppings if attr[0:2] == 'Tr'][10]
[len(c) for c in len_sorted_comps][0]

# len_sorted_comps[0]

716347

In [119]:
sorted(list(transactions_graph.degree(len_sorted_comps[0])), key=lambda x: x[1], reverse=True)[:100]

[('Addr287.0', 520481),
 ('Addr1299.0', 46335),
 ('Addr1325.0', 42751),
 ('Addr1204.0', 42020),
 ('Addr1264.0', 39870),
 ('Addr1330.0', 26287),
 ('Addr1315.0', 23078),
 ('Addr1441.0', 20827),
 ('Addr1272.0', 20141),
 ('Addr1123.0', 16105),
 ('Addr1126.0', 15243),
 ('Addr1184.0', 15160),
 ('Addr1337.0', 15149),
 ('Addr1191.0', 14979),
 ('C17919', 14932),
 ('C19500', 14162),
 ('Addr1181.0', 13856),
 ('C115885', 10361),
 ('C117188', 10344),
 ('Addr1143.0', 9806),
 ('Addr1476.0', 9478),
 ('Addr1310.0', 8486),
 ('Addr1472.0', 8478),
 ('Addr1327.0', 8425),
 ('Addr1512.0', 8268),
 ('Addr1387.0', 8187),
 ('C115066', 7945),
 ('Addr1433.0', 7831),
 ('Addr1231.0', 7605),
 ('C112695', 7091),
 ('Addr1485.0', 6816),
 ('C112544', 6773),
 ('C16019', 6771),
 ('Addr1269.0', 6404),
 ('C12803', 6141),
 ('Addr1205.0', 5725),
 ('C17585', 5334),
 ('Addr1225.0', 5323),
 ('Addr1251.0', 5216),
 ('C110616', 5172),
 ('C112839', 5129),
 ('Addr1494.0', 5065),
 ('Addr1220.0', 5041),
 ('Addr1226.0', 4867),
 ('Addr149